<a href="https://colab.research.google.com/github/TomkerDev/-Projet-Analyse-Comparative-du-March-Web-Scraping-/blob/main/%F0%9F%8F%97%EF%B8%8F_Projet_Analyse_Comparative_du_March%C3%A9_(Web_Scraping).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

.

# 🏗️ Projet : Analyse Comparative du Marché (Web Scraping)
L'objectif est d'extraire automatiquement les noms, les prix et les caractéristiques des produits pour analyser la concurrence

1. Les Outils (La Stack Technique)
BeautifulSoup : Pour extraire les données du code HTML.

- Requests : Pour envoyer des requêtes aux sites web.

- Pandas : Pour organiser les données dans un tableau propre.

- Google Colab : Pour coder et tester rapidement.

In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import datetime
from fpdf import FPDF
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
import time

## 🛠️ Étape 1 : Le code de base (Pas à pas)

Nous allons essayer de scraper un site "école" (conçu pour l'entraînement) afin d'éviter les blocages de sécurité.

In [11]:
def executer_scraping_1():
    url = "http://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    articles = soup.find_all('article', class_='product_pod')

    data = []
    for article in articles:
        title = article.h3.a['title']
        price = article.find('p', class_='price_color').text
        data.append([title, price])

    df = pd.DataFrame(data, columns=['Titre', 'Prix'])
    # Nettoyage rapide pour les stats
    df['Prix_Num'] = df['Prix'].str.replace('£', '').astype(float)
    return df

## 🛠️ Le Code pour Scraper Plusieurs Pages

Pour scraper plusieurs pages, nous allons utiliser une boucle for qui va changer l'URL à chaque itération.

In [13]:
def executer_scraping_many():
    all_data = []
    # On définit le nombre de pages à parcourir (ex: les 3 premières pages)
    nombre_de_pages = 3

    for page in range(1, nombre_de_pages + 1):
        print(f"Extraction de la page {page}...")
        url = f"http://books.toscrape.com/catalogue/category/books/sequential-art_5/page-{page}.html"

        response = requests.get(url)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            articles = soup.find_all('article', class_='product_pod')

            for article in articles:
                title = article.h3.a['title']
                price = article.find('p', class_='price_color').text
                all_data.append([title, price])

            # Pause respectueuse pour le serveur
            time.sleep(1)
        else:
            print(f"Fin des pages ou erreur à la page {page}")
            break

    df = pd.DataFrame(all_data, columns=['Titre', 'Prix'])
    # Nettoyage pour transformer "£51.77" en 51.77 (float)
    df['Prix_Num'] = df['Prix'].str.replace('£', '').astype(float)

    print(f"Total collecté : {len(df)} produits.")
    return df

## ÉTAPE 2 : FONCTION DE GÉNÉRATION PDF
Ce script va calculer des statistiques clés et les écrire dans un fichier PDF.

In [17]:
# --- CONFIGURATION DE L'ENTREPRISE (Personnalisable) ---
ENTREPRISE_NOM = "Tete Roh Studio - Data Analytics"
ANALYSTE_NOM = "Ing. Tomté Hassane"
LOGO_PATH = "distribution_prix.png" # On peut réutiliser le graphique ou un vrai logo

class ProfessionalPDF(FPDF):
    def header(self):
        # En-tête avec bannière grise
        self.set_fill_color(31, 56, 100) # Bleu Marine
        self.rect(0, 0, 210, 40, 'F')
        self.set_text_color(255, 255, 255)
        self.set_font('Arial', 'B', 20)
        self.cell(0, 20, f"{ENTREPRISE_NOM}", 0, 1, 'L')
        self.set_font('Arial', 'I', 12)
        self.cell(0, 5, "Rapport Stratégique de Veille Concurrentielle", 0, 1, 'L')
        self.ln(15)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.set_text_color(128, 128, 128)
        self.cell(0, 10, f'Page {self.page_no()} | Rapport confidentiel - {ENTREPRISE_NOM}', 0, 0, 'C')

def generer_rapport_riche(df):
    pdf = ProfessionalPDF()
    pdf.add_page()
    pdf.set_text_color(0, 0, 0)

    # 1. Informations de synthèse
    date_gen = datetime.datetime.now().strftime("%d/%m/%Y %H:%M")
    pdf.set_font("Arial", 'B', 12)
    pdf.cell(0, 10, f"Date d'extraction : {date_gen}", ln=True)
    pdf.cell(0, 10, f"Responsable Analyse : {ANALYSTE_NOM}", ln=True)
    pdf.ln(5)

    # 2. Section Tableaux de Bord (KPIs)
    pdf.set_fill_color(240, 240, 240)
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, " I. Indicateurs Clés de Performance (KPI)", ln=True, fill=True)
    pdf.ln(5)

    pdf.set_font("Arial", size=11)
    # On crée une structure en colonnes
    pdf.cell(90, 10, f"Volume de données : {len(df)} articles", 1)
    pdf.cell(90, 10, f"Prix Moyen : {df['Prix_Num'].mean():.2f} GBP", 1, ln=True)
    pdf.cell(90, 10, f"Prix Minimum : {df['Prix_Num'].min():.2f} GBP", 1)
    pdf.cell(90, 10, f"Prix Maximum : {df['Prix_Num'].max():.2f} GBP", 1, ln=True)
    pdf.ln(10)

    # 3. Section Visualisation (L'IMAGE !)
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, " II. Distribution Visuelle des Prix", ln=True, fill=True)
    pdf.ln(5)
    # Insertion de l'histogramme
    pdf.image('distribution_prix.png', x=15, w=180)
    pdf.ln(5)

    # 4. Conclusion Marketing
    pdf.add_page() # Nouvelle page pour la conclusion
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, " III. Analyse et Recommandations", ln=True, fill=True)
    pdf.ln(5)
    pdf.set_font("Arial", size=11)

    conclusion = (
        "L'analyse automatisée montre une concentration des prix dans la tranche moyenne. "
        "Pour optimiser la compétitivité, il est recommandé de surveiller les produits "
        f"dépassant le prix maximum de {df['Prix_Num'].max():.2f} GBP."
    )
    pdf.multi_cell(0, 10, conclusion)

    filename = "Rapport_Analytique_Entreprise.pdf"
    pdf.output(filename)
    return filename

## 2. Envoyer le rapport par Email (SMTP)

- A un seul destinateur

In [22]:

# --- CONFIGURATION RÉELLE ---
EMAIL_EXPEDITEUR = "tomtehassane33@gmail.com"  # Ton adresse Gmail
MOT_DE_PASSE = "mnto icfs slrc nqbe"      # Ton code de 16 caractères Google
EMAIL_DESTINATAIRE = "destination@gmail.com"

def envoyer_rapport_reel(fichier_pdf):
    print(f"Connexion au serveur SMTP pour l'envoi à {EMAIL_DESTINATAIRE}...")

    try:
        # Création de l'objet email
        msg = MIMEMultipart()
        msg['From'] = EMAIL_EXPEDITEUR
        msg['To'] = EMAIL_DESTINATAIRE
        msg['Subject'] = f"📊 Rapport d'Analyse - {datetime.datetime.now().strftime('%d/%m/%Y')}"

        corps = f"""Bonjour,

Veuillez trouver ci-joint le rapport de veille automatisé généré par le système Tete Roh Studio.

Détails de l'exécution :
- Analyste : Tomté Hassane
- Statut : Succès
- Fichier : {fichier_pdf}

Cordialement,
L'automate Data Engineering."""

        msg.attach(MIMEText(corps, 'plain'))

        # Pièce jointe (le PDF riche)
        with open(fichier_pdf, "rb") as attachment:
            part = MIMEBase('application', 'octet-stream')
            part.set_payload(attachment.read())
            encoders.encode_base64(part)
            part.add_header('Content-Disposition', f"attachment; filename= {fichier_pdf}")
            msg.attach(part)

        # Envoi via les serveurs de Google
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls() # Sécurisation de la connexion
        server.login(EMAIL_EXPEDITEUR, MOT_DE_PASSE)
        server.send_message(msg)
        server.quit()

        print("✅ Email envoyé avec succès !")

    except Exception as e:
        print(f"❌ Erreur lors de l'envoi : {e}")


- A plusieurs  destinateurs

In [ ]:


# Liste de tes décideurs
LISTE_DECIDEURS = [
    "decideur1@exemple.com",
    "decideur2@exemple.com",
    "decideur3@exemple.com"
]

def envoyer_rapport_aux_decideurs(fichier_pdf, liste_emails):
    try:
        # Connexion au serveur une seule fois
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(EMAIL_EXPEDITEUR, MOT_DE_PASSE)

        for destinataire in liste_emails:
            print(f"Envoi en cours vers : {destinataire}...")

            msg = MIMEMultipart()
            msg['From'] = EMAIL_EXPEDITEUR
            msg['To'] = destinataire
            msg['Subject'] = f"📊 Rapport Stratégique - Tete Roh Studio - {datetime.datetime.now().strftime('%d/%m/%Y')}"

            corps = f"Bonjour,\n\nVeuillez trouver ci-joint le rapport d'analyse de marché automatisé.\n\nCordialement,\nTomté Hassane\nData Engineer"
            msg.attach(MIMEText(corps, 'plain'))

            with open(fichier_pdf, "rb") as attachment:
                part = MIMEBase('application', 'octet-stream')
                part.set_payload(attachment.read())
                encoders.encode_base64(part)
                part.add_header('Content-Disposition', f"attachment; filename= {fichier_pdf}")
                msg.attach(part)

            server.send_message(msg)
            print(f"✅ Succès pour {destinataire}")

        server.quit()

    except Exception as e:
        print(f"❌ Erreur générale : {e}")



## 1. Planifier la tâche (Scheduling)

In [23]:
def job():
    print("Démarrage du pipeline haute performance...")
    donnees = executer_scraping_many() # Ta fonction multi-pages
    fichier_pdf = generer_rapport_riche(donnees) # Le nouveau rapport riche
    envoyer_rapport_reel(fichier_pdf)
    # 3. Envoi groupé
    # envoyer_rapport_aux_decideurs(fichier_pdf, LISTE_DECIDEURS)
    print(f"Félicitations ! Ton rapport riche '{fichier_pdf}' est prêt.")
# --- TEST IMMÉDIAT ---
job()

Démarrage du pipeline haute performance...
Extraction de la page 1...
Extraction de la page 2...
Extraction de la page 3...
Total collecté : 60 produits.
Connexion au serveur SMTP pour l'envoi à akamrabe@gmail.com...
✅ Email envoyé avec succès !
Félicitations ! Ton rapport riche 'Rapport_Analytique_Entreprise.pdf' est prêt.


# 🏁 Conclusion & Perspectives du Projet
Ce projet a permis de mettre en place un pipeline de données complet, allant de l'extraction brute à la communication décisionnelle.

✅ Objectifs Atteints :
- Collecte de données (Scraping) : Extraction automatisée sur plusieurs pages pour constituer un dataset représentatif.

- Ingénierie des données : Nettoyage rigoureux des formats monétaires et gestion des valeurs manquantes.

- Visualisation Analytique : Création de graphiques de distribution pour identifier les tendances du marché.

- Reporting & Automatisation : Génération d'un rapport PDF professionnel et envoi multi-destinataires via le protocole SMTP.

## 📈 Business Insights :
L'analyse montre une grande diversité de prix, permettant d'identifier des opportunités de positionnement concurrentiel. Grâce à l'automatisation, les décideurs reçoivent désormais une information fraîche, précise et visuelle, sans intervention humaine répétitive.

## 🚀 Évolutions futures (Next Steps) :
I- ntégration d'une Base de Données : Stocker les résultats dans une base SQL (PostgreSQL) pour suivre l'évolution des prix dans le temps.

- Dashboard Interactif : Connecter ce pipeline à un outil comme Power BI ou Streamlit pour une exploration en temps réel.

- Analyse de Sentiment : Scraper les commentaires des utilisateurs pour coupler l'analyse des prix à la satisfaction client.